Importações

Só precisamos de duas bibliotecas: `numpy` para gerar os números aleatórios e fazer as contas, e `pandas` para organizar tudo em um DataFrame no final.

In [18]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 100)

 A assinatura da função

```python
def gerar_dados_emprestimo(n=450, semente=42):
```

Dois parâmetros, ambos com valor padrão:

- **`n`** — quantas linhas (solicitações de empréstimo) o dataset vai ter. Padrão: 450.
- **`semente`** — a semente do gerador de números aleatórios. É o que torna a função **reprodutível**: chamando com a mesma semente, sempre saem os mesmos números, na mesma ordem.

Para acompanhar o restante do notebook, vamos fixar os dois valores exatamente como o material da aula usa.

In [ ]:
n = 3000
semente = 42

Linha 1 — `rng = np.random.default_rng(semente)`

Cria um **gerador de números aleatórios** (`Generator`) inicializado com a semente. É a forma recomendada atualmente pelo NumPy de gerar aleatoriedade reprodutível — substitui o antigo `np.random.seed(...)` seguido de `np.random.normal(...)` etc.

A partir daqui, todo número aleatório da função sai desse `rng`, sempre na mesma ordem em que é chamado. Se a ordem das chamadas mudar, os números que saem depois também mudam — por isso é importante nunca reordenar as linhas da função.

In [15]:
rng = np.random.default_rng(semente)
rng

Generator(PCG64) at 0x28908CED380

Linha 2 — `idade = rng.normal(38, 12, n).clip(18, 75).round().astype(float)`

Lida da esquerda para a direita, é uma sequência de quatro operações encadeadas:

1. `rng.normal(38, 12, n)` — sorteia `n` valores de uma **distribuição normal** (gaussiana) com média 38 e desvio-padrão 12. É o formato clássico de "sino": a maioria das idades fica perto de 38, poucas ficam muito longe.
2. `.clip(18, 75)` — qualquer valor sorteado abaixo de 18 vira 18, e acima de 75 vira 75. Isso evita idades absurdas (negativas, ou 130 anos) que a cauda da normal poderia gerar, mantendo a faixa plausível para quem solicita um empréstimo.
3. `.round()` — arredonda para o inteiro mais próximo (não faz sentido ter 38,7 anos).
4. `.astype(float)` — mesmo já sendo um número "inteiro" depois do `round()`, o tipo continua `float`. Isso é proposital: mais adiante algumas idades vão virar `NaN` (valor ausente), e `NaN` só existe em colunas `float` — uma coluna `int` do NumPy não aceita `NaN`.

In [16]:
idade = rng.normal(38, 12, n).clip(18, 75).round().astype(float)
print(idade[:5])
print("min:", idade.min(), " max:", idade.max(), " media:", round(idade.mean(), 1))

[42. 26. 47. 49. 18.]
min: 18.0  max: 73.0  media: 38.0


Linhas 3 e 4 — `score_credito` e `tempo_emprego`

A mesma receita da idade, repetida com outros parâmetros:

```python
score_credito = rng.normal(650, 80, n).clip(300, 850).round().astype(float)
tempo_emprego = rng.normal(7, 5, n).clip(0, 35).round().astype(float)
```

- **`score_credito`** — normal com média 650 e desvio 80, limitado entre 300 e 850 (a mesma faixa usada por modelos de score de crédito conhecidos, como o FICO).
- **`tempo_emprego`** — normal com média 7 anos e desvio 5, limitado entre 0 (acabou de começar) e 35 anos.

Repare que cada chamada a `rng.normal(...)` consome uma nova sequência de números aleatórios do gerador — é por isso que `idade`, `score_credito` e `tempo_emprego` saem com valores diferentes entre si, mesmo vindo do mesmo `rng`.

In [20]:
score_credito = rng.normal(650, 80, n).clip(300, 850).round().astype(float)
tempo_emprego = rng.normal(7, 5, n).clip(0, 35).round().astype(float)

print("score_credito:", score_credito[:10])
print("tempo_emprego:", tempo_emprego[:10])

score_credito: [640. 678. 806. 816. 656. 663. 736. 582. 677. 648.]
tempo_emprego: [ 9.  8.  6. 15.  4. 17.  6.  1.  7.  2.]
score_credito: [647. 685. 626. 617. 657. 496. 688. 650. 653. 727.]
tempo_emprego: [ 2.  7. 10.  0. 12.  1.  9.  8. 12. 10.]


Linha 5 — `renda = rng.lognormal(mean=8.15, sigma=0.45, size=n)`

Aqui a distribuição muda: em vez de `normal`, usamos `lognormal` (**log-normal**). A diferença importa: renda, na vida real, não segue uma curva em sino simétrica — a maioria das pessoas ganha perto de um valor "típico", mas existe uma cauda longa de rendas altas (bem menos gente, mas concentrando valores bem maiores). A log-normal reproduz exatamente essa assimetria: é sempre positiva e tem uma cauda longa à direita.

Os parâmetros `mean=8.15` e `sigma=0.45` não são a média e o desvio da renda em reais — são a média e o desvio do **logaritmo** da renda (por isso "log-normal"). Na prática, isso gera valores concentrados na faixa de alguns milhares de reais, com uma cauda de valores maiores.

In [22]:
renda = rng.lognormal(mean=8.15, sigma=0.45, size=n)
print(renda[:10].round(2))
print("mediana:", round(np.median(renda), 2), " min:", round(renda.min(), 2), " max:", round(renda.max(), 2))

[3090.86 4460.53 1936.53 4058.96 8191.31 4271.75 2923.99 2756.74 2879.43
 3089.35]
mediana: 3305.31  min: 915.96  max: 12854.0
[3385.37 4759.7  3348.79 2073.32 3249.87 3570.28 3647.22 3725.15 5528.74
 2124.48]
mediana: 3489.41  min: 1009.5  max: 13722.46


Linhas 6 e 7 — criando outliers propositais em `renda`

```python
idx_outlier = rng.choice(n, size=5, replace=False)
renda[idx_outlier] *= rng.uniform(6, 10, size=5)
```

- `rng.choice(n, size=5, replace=False)` — sorteia **5 posições (índices) diferentes**, sem repetição (`replace=False`), dentre as `n` linhas do dataset. São as linhas que vão virar outliers.
- `renda[idx_outlier] *= rng.uniform(6, 10, size=5)` multiplica a renda dessas 5 linhas por um fator aleatório entre 6 e 10. Ou seja, essas 5 pessoas passam a ter uma renda de 6 a 10 vezes maior do que o valor original sorteado — valores bem acima do padrão do restante do dataset.

Esses outliers propositais são exatamente o que a Aula 4 usa para demonstrar a regra do IQR (intervalo interquartil) na detecção de valores atípicos.

In [24]:
idx_outlier = rng.choice(n, size=5, replace=False)
print("indices sorteados como outliers:", idx_outlier)

renda[idx_outlier] *= rng.uniform(6, 10, size=5)
print("renda dessas linhas depois de multiplicar:")
print(renda[idx_outlier].round(2))

indices sorteados como outliers: [ 12  44 141 263 273]
renda dessas linhas depois de multiplicar:
[13815.37 13972.91 34158.62 23850.36 54066.47]
indices sorteados como outliers: [201  59 198 429   6]
renda dessas linhas depois de multiplicar:
[59032.28 46980.48 16661.17 81405.89 26258.89]


Linha 8 `renda = renda.round(2)`

Arredonda a renda para 2 casas decimais (centavos) — o mesmo cuidado de formatação que já vimos com `idade`, mas aqui mantendo casas decimais porque renda é um valor monetário.

In [26]:
renda = renda.round(2)
print(renda[:10])

[3385.37 4759.7  3348.79 2073.32 3249.87 3570.28 3647.22 3725.15 5528.74
 2124.48]
[ 3385.37  4759.7   3348.79  2073.32  3249.87  3570.28 26258.89  3725.15
  5528.74  2124.48]


Linhas 9 e 10 — a coluna categórica `cidade`

```python
cidades = np.array(["SP", "RJ", "MG", "Outra"])
cidade = rng.choice(cidades, size=n, p=[0.42, 0.23, 0.15, 0.20])
```

- `cidades` — um array com as quatro categorias possíveis.
- `rng.choice(cidades, size=n, p=[...])` — sorteia, para cada uma das `n` linhas, uma das quatro categorias. O parâmetro `p` dá o **peso de cada categoria**: 42% de chance de "SP", 23% de "RJ", 15% de "MG" e 20% de "Outra" (os quatro pesos somam 1,0 = 100%). Sem o `p`, o `choice` sortearia as quatro categorias com a mesma probabilidade (25% cada); com `p`, reproduzimos uma distribuição mais realista, com "SP" concentrando a maior fatia.

Essa é a única coluna do dataset que nasce como texto (`string`), e não como número — por isso ela vai precisar de codificação (one-hot) mais adiante, no preparo dos dados.

In [28]:
cidades = np.array(["SP", "RJ", "MG", "Outra"])
cidade = rng.choice(cidades, size=n, p=[0.42, 0.23, 0.15, 0.20])

print(cidade[:10])
valores, contagens = np.unique(cidade, return_counts=True)
for v, c in zip(valores, contagens):
    print(f"  {v}: {c} ({100*c/n:.1f}%)")

['SP' 'RJ' 'SP' 'Outra' 'Outra' 'RJ' 'Outra' 'RJ' 'SP' 'RJ']
  MG: 61 (13.6%)
  Outra: 92 (20.4%)
  RJ: 110 (24.4%)
  SP: 187 (41.6%)
['SP' 'Outra' 'SP' 'RJ' 'SP' 'RJ' 'SP' 'Outra' 'SP' 'RJ']
  MG: 74 (16.4%)
  Outra: 88 (19.6%)
  RJ: 97 (21.6%)
  SP: 191 (42.4%)


Linhas 11 a 13 — construindo a probabilidade de aprovação

```python
renda_c = np.clip(renda, None, np.percentile(renda, 95))
z = (
    0.05 * (score_credito - 650)
    + 0.0008 * (renda_c - renda_c.mean())
    + 0.12 * (tempo_emprego - 7)
    + rng.normal(0, 0.35, n)
)
prob_aprovado = 1 / (1 + np.exp(-z))
```

Por partes:

- **`renda_c = np.clip(renda, None, np.percentile(renda, 95))`** — cria uma cópia "contida" da renda, cortando (sem alterar a `renda` original) tudo o que está acima do percentil 95 no valor desse percentil. Isso evita que os 5 outliers criados antes dominem sozinhos o cálculo que vem a seguir — eles continuam outliers na coluna `renda` do dataset final, mas não distorcem a probabilidade de aprovação de forma exagerada. `None` no primeiro limite do `clip` significa "não cortar por baixo".
- **`z = ...`** — uma combinação linear, no mesmo espírito de uma regressão logística: quanto maior o `score_credito` (acima de 650), maior `z`; quanto maior a `renda_c` (acima da média), maior `z`; quanto maior o `tempo_emprego` (acima de 7 anos), maior `z`. Cada coeficiente (`0.05`, `0.0008`, `0.12`) foi calibrado para que os três atributos pesem de forma parecida, apesar de estarem em escalas bem diferentes (score na casa das centenas, renda na casa dos milhares, tempo em unidades). O termo `rng.normal(0, 0.35, n)` adiciona ruído aleatório — sem ele, a aprovação seria 100% determinística a partir desses três atributos, o que não é realista.
- **`prob_aprovado = 1 / (1 + np.exp(-z))`** — é a **função sigmoide** (logística), que comprime qualquer número real `z` para o intervalo entre 0 e 1. `z` grande e positivo → probabilidade perto de 1; `z` grande e negativo → probabilidade perto de 0; `z = 0` → probabilidade exatamente 0,5. É a mesma função usada na regressão logística para transformar uma pontuação em uma probabilidade.

In [44]:
renda_c = np.clip(renda, None, np.percentile(renda, 95))
print("maior valor de renda_c (limitado):", round(renda_c.max(), 2))
print("maior valor de renda (sem limite):", round(renda.max(), 2))

z = (
    0.05 * (score_credito - 650)
    + 0.0008 * (renda_c - renda_c.mean())
    + 0.12 * (tempo_emprego - 7)
    + rng.normal(0, 0.35, n)
)
prob_aprovado = 1 / (1 + np.exp(-z))

print("\nz (primeiras 10 linhas):", z[:10].round(3))
print("prob_aprovado (primeiras 10 linhas):", prob_aprovado[:10].round(3))

maior valor de renda_c (limitado): 8099.18
maior valor de renda (sem limite): 81405.89

z (primeiras 10 linhas): [-1.374  2.033 -1.16  -4.153  0.467 -9.168  5.397  0.216  2.251  2.499]
prob_aprovado (primeiras 10 linhas): [0.202 0.884 0.239 0.015 0.615 0.    0.995 0.554 0.905 0.924]


Linha 14 — `aprovado = (rng.uniform(0, 1, n) < prob_aprovado).astype(int)`

Transforma cada probabilidade em uma decisão de 0 ou 1, através de um **sorteio de Bernoulli**: para cada linha, sorteia um número uniforme entre 0 e 1; se esse número sorteado for **menor** que a probabilidade de aprovação daquela linha, `aprovado` vira `True` (depois convertido para `1` pelo `.astype(int)`); caso contrário, vira `0`.

Isso é diferente de simplesmente arredondar a probabilidade (`prob_aprovado > 0.5`): com o sorteio, uma linha com `prob_aprovado = 0.9` tem 90% de chance de ser aprovada, mas ainda pode, por acaso, sair como reprovada — o que é mais realista do que uma regra 100% determinística.

In [32]:
aprovado = (rng.uniform(0, 1, n) < prob_aprovado).astype(int)

print(aprovado[:10])
print("proporcao aprovada:", round(aprovado.mean(), 4))

[0 1 0 0 0 0 1 0 1 1]
proporcao aprovada: 0.4978
[0 1 0 0 1 0 1 1 1 1]
proporcao aprovada: 0.4689


Linhas 15 a 22 — montando o DataFrame

```python
df = pd.DataFrame({
    "idade": idade,
    "renda": renda,
    "tempo_emprego": tempo_emprego,
    "score_credito": score_credito,
    "cidade": cidade,
    "aprovado": aprovado,
})
```

Até aqui, `idade`, `renda`, `tempo_emprego`, `score_credito`, `cidade` e `aprovado` são seis arrays NumPy separados, todos com o mesmo comprimento `n`. `pd.DataFrame({...})` os junta em uma única tabela, usando as chaves do dicionário como nome de cada coluna. É só a partir desta linha que existe, de fato, um "dataset" — antes disso, eram só variáveis soltas.

In [34]:
df = pd.DataFrame({
    "idade": idade,
    "renda": renda,
    "tempo_emprego": tempo_emprego,
    "score_credito": score_credito,
    "cidade": cidade,
    "aprovado": aprovado,
})

df.head(10)

,idade,renda,tempo_emprego,score_credito,cidade,aprovado
0,42.0,3385.37,2.0,647.0,SP,0
1,26.0,4759.70,7.0,685.0,Outra,1
2,47.0,3348.79,10.0,626.0,SP,0
3,49.0,2073.32,0.0,617.0,RJ,0
4,18.0,3249.87,12.0,657.0,SP,0
5,22.0,3570.28,1.0,496.0,RJ,0
6,40.0,26258.89,9.0,688.0,SP,1
7,34.0,3725.15,8.0,650.0,Outra,0
8,38.0,5528.74,12.0,653.0,SP,1
9,28.0,2124.48,10.0,727.0,RJ,1


,idade,renda,tempo_emprego,score_credito,cidade,aprovado
0,42.0,3385.37,2.0,647.0,SP,0
1,26.0,4759.70,7.0,685.0,Outra,1
2,47.0,3348.79,10.0,626.0,SP,0
3,49.0,2073.32,0.0,617.0,RJ,0
4,18.0,3249.87,12.0,657.0,SP,1
5,22.0,3570.28,1.0,496.0,RJ,0
6,40.0,26258.89,9.0,688.0,SP,1
7,34.0,3725.15,8.0,650.0,Outra,1
8,38.0,5528.74,12.0,653.0,SP,1
9,28.0,2124.48,10.0,727.0,RJ,1


Linhas 23 a 25 — sorteando quais linhas vão ficar com valor ausente

```python
mask_idade = rng.uniform(0, 1, n) < 0.08
mask_renda = rng.uniform(0, 1, n) < 0.10
mask_cidade = rng.uniform(0, 1, n) < 0.06
```

Três **máscaras booleanas** (arrays de `True`/`False`, um valor por linha), uma para cada coluna que vai ganhar valores ausentes. A lógica é a mesma do sorteio de `aprovado`: sorteia um número uniforme entre 0 e 1 para cada linha e marca `True` quando esse número fica abaixo do limite — `0.08` para idade (≈8% das linhas), `0.10` para renda (≈10%), `0.06` para cidade (≈6%). Como os três limites são diferentes, cada coluna acaba com uma proporção diferente de ausentes — e como os sorteios são independentes entre si, uma mesma linha pode (raramente) ficar com mais de uma coluna ausente ao mesmo tempo.

In [45]:
mask_idade = rng.uniform(0, 1, n) < 0.08
mask_renda = rng.uniform(0, 1, n) < 0.10
mask_cidade = rng.uniform(0, 1, n) < 0.06

print("linhas marcadas para ficar sem idade:", mask_idade.sum())
print("linhas marcadas para ficar sem renda:", mask_renda.sum())
print("linhas marcadas para ficar sem cidade:", mask_cidade.sum())

linhas marcadas para ficar sem idade: 31
linhas marcadas para ficar sem renda: 56
linhas marcadas para ficar sem cidade: 37


Linhas 26 a 28 — aplicando os valores ausentes

```python
df.loc[mask_idade, "idade"] = np.nan
df.loc[mask_renda, "renda"] = np.nan
df.loc[mask_cidade, "cidade"] = None
```

`df.loc[mascara, "coluna"] = valor` seleciona, dentro da coluna indicada, só as linhas onde a máscara é `True`, e substitui o valor original por `np.nan` (para as colunas numéricas `idade` e `renda`) ou `None` (para a coluna de texto `cidade` — o pandas converte `None` para o equivalente de ausente também em colunas de texto). É só depois dessas três linhas que o dataset passa a ter, de fato, valores ausentes — até aqui, ele estava completo.

In [38]:
df.loc[mask_idade, "idade"] = np.nan
df.loc[mask_renda, "renda"] = np.nan
df.loc[mask_cidade, "cidade"] = None

print(df.isna().sum())

idade            37
renda            42
tempo_emprego     0
score_credito     0
cidade           27
aprovado          0
dtype: int64
idade            75
renda            78
tempo_emprego     0
score_credito     0
cidade           58
aprovado          0
dtype: int64


Linha 29 — `return df`

Devolve o DataFrame completo, já com os valores ausentes aplicados. É esse `df` que o restante da Aula 4 caracteriza (Parte 1) e prepara (Parte 2).

### A função inteira, junta

Reunindo tudo o que acabamos de construir passo a passo:

In [40]:
def gerar_dados_emprestimo(n=450, semente=42):
    rng = np.random.default_rng(semente)

    idade = rng.normal(38, 12, n).clip(18, 75).round().astype(float)
    score_credito = rng.normal(650, 80, n).clip(300, 850).round().astype(float)
    tempo_emprego = rng.normal(7, 5, n).clip(0, 35).round().astype(float)

    renda = rng.lognormal(mean=8.15, sigma=0.45, size=n)
    idx_outlier = rng.choice(n, size=5, replace=False)
    renda[idx_outlier] *= rng.uniform(6, 10, size=5)
    renda = renda.round(2)

    cidades = np.array(["SP", "RJ", "MG", "Outra"])
    cidade = rng.choice(cidades, size=n, p=[0.42, 0.23, 0.15, 0.20])

    renda_c = np.clip(renda, None, np.percentile(renda, 95))
    z = (
        0.05 * (score_credito - 650)
        + 0.0008 * (renda_c - renda_c.mean())
        + 0.12 * (tempo_emprego - 7)
        + rng.normal(0, 0.35, n)
    )
    prob_aprovado = 1 / (1 + np.exp(-z))
    aprovado = (rng.uniform(0, 1, n) < prob_aprovado).astype(int)

    df = pd.DataFrame({
        "idade": idade,
        "renda": renda,
        "tempo_emprego": tempo_emprego,
        "score_credito": score_credito,
        "cidade": cidade,
        "aprovado": aprovado,
    })

    mask_idade = rng.uniform(0, 1, n) < 0.08
    mask_renda = rng.uniform(0, 1, n) < 0.10
    mask_cidade = rng.uniform(0, 1, n) < 0.06
    df.loc[mask_idade, "idade"] = np.nan
    df.loc[mask_renda, "renda"] = np.nan
    df.loc[mask_cidade, "cidade"] = None

    return df


df = gerar_dados_emprestimo(n=450, semente=42)
print(df.shape)
df.head()

(450, 6)


,idade,renda,tempo_emprego,score_credito,cidade,aprovado
0,NaN,3090.86,9.0,640.0,Outra,0
1,26.0,4460.53,8.0,678.0,SP,1
2,47.0,1936.53,6.0,806.0,SP,1
3,49.0,4058.96,15.0,816.0,SP,1
4,18.0,8191.31,4.0,656.0,SP,1


(450, 6)


,idade,renda,tempo_emprego,score_credito,cidade,aprovado
0,NaN,3090.86,9.0,640.0,Outra,0
1,26.0,4460.53,8.0,678.0,SP,1
2,47.0,1936.53,6.0,806.0,SP,1
3,49.0,4058.96,15.0,816.0,SP,1
4,18.0,8191.31,4.0,656.0,SP,1


Salvando o dataset **sem tratamento** (bruto)

Antes de qualquer preparo, salvamos o DataFrame exatamente como a função devolveu: com os valores ausentes em `idade`, `renda` e `cidade`, e com os outliers em `renda` ainda presentes. Esse é o arquivo "cru", equivalente ao que a Parte 1 da Aula 4 caracteriza.

In [42]:
df.to_csv("emprestimos_bruto.csv", index=False)
print("salvo: emprestimos_bruto.csv")
print(df.isna().sum())

salvo: emprestimos_bruto.csv
idade            75
renda            78
tempo_emprego     0
score_credito     0
cidade           58
aprovado          0
dtype: int64
salvo: emprestimos_bruto.csv
idade            41
renda            42
tempo_emprego     0
score_credito     0
cidade           33
aprovado          0
dtype: int64


Tratamento da base e salvamento da versão tratada

*(a explicação linha a linha desta parte fica para depois — aqui só o código, reaproveitando o `ColumnTransformer` do Bloco 4 de `lab04_parte1_demo.py`)*

In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

colunas_num = ["idade", "renda", "tempo_emprego", "score_credito"]
colunas_cat = ["cidade"]

# Prepara e aplica as transformacoes
prep = ColumnTransformer([
    # Todas as linhas "num" ficam com a media
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), colunas_num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), colunas_cat),
])

X = df.drop(columns="aprovado")
y = df["aprovado"]

X_tratado = prep.fit_transform(X)

#Obtem os nome das colunas Cidades transformadas
colunas_finais = colunas_num + list(
    prep.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(colunas_cat)
)
df_tratado = pd.DataFrame(X_tratado, columns=colunas_finais)
df_tratado["aprovado"] = y.values

df_tratado.to_csv("emprestimos_tratado.csv", index=False)
print("salvo: emprestimos_tratado.csv")
print(df_tratado.shape)
df_tratado.head()

salvo: emprestimos_tratado.csv
(450, 9)


,idade,renda,tempo_emprego,score_credito,cidade_MG,cidade_Outra,cidade_RJ,cidade_SP,aprovado
0,-0.067521,-0.256076,0.378543,-0.053321,0.0,1.0,0.0,0.0,0
1,-1.099086,0.245333,0.161267,0.414744,0.0,0.0,0.0,1.0,1
2,0.870266,-0.678654,-0.273285,1.991384,0.0,0.0,0.0,1.0,1
3,1.057823,0.098326,1.682200,2.114559,0.0,0.0,0.0,1.0,1
4,-1.849315,1.611099,-0.707837,0.143759,0.0,0.0,0.0,1.0,1
